In [ ]:
import socket
from pathlib import Path
import numpy as np
import pandas as pd
import torch

np.random.seed(0)
torch.manual_seed(0)

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mc
import seaborn as sns
from IPython.display import display
from scipy.stats import pearsonr

%matplotlib inline

import pickle

from vi_rnn.saving import load_model, CPU_Unpickler
from vi_rnn.data_utils import make_all_trials
from fig_utils.transformed_rnn import transformed_rnn



In [ ]:
from fig_utils.perturbation import (
    compute_perturbation_distance_stats,
    generate_w_perturb_x,
    position_latent_indices,
)
from fig_utils.plots import (
    plot_basis_2d_subspaces,
    plot_distance_moved_vs_class_mean,
    plot_pearson_r_by_macaque,
    plot_perturbation_latent_snapshots_attractor,
    position_slice_at_time,
)

In [ ]:
hostname = socket.gethostname()
print("hostname:", hostname)

if hostname == "MatthijsDesktop":
    out_dir = Path("/home/matthijs/swm_rnn/final_models/macaque")
    path = "/home/matthijs/swm_rnn/data/"

else:
    out_dir = Path("/Users/matthijs/swm_rnn/final_models/macaque")
    path = str(Path.cwd().parent / "data") + "/"


model_dirs = [
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_28_T_05_34_01",
    "SWM_low_rank_one_to_one_dim_z_64_date_2026_05_01_T_22_03_56",
    "SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_18_03_12",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_17_02_13",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_16_30_36",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_16_26_38",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_27_25",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_30_22",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_20_35",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_13_05",
]

In [ ]:
# load the basis dataframe
df_basii = pickle.load(open("../data/processed/df_basii.pkl", "rb"))

In [ ]:
# --- controls ---

# --- task / basis ---
n_pos = 3
n_pcs_time = 2
n_stim = 6
t_decode = -1

# --- perturbation setup ---
pos_perturb = [0, 1, 2]
z1s = [n_pcs_time + 2 * p for p in pos_perturb]
z2s = [n_pcs_time + 2 * p + 1 for p in pos_perturb]
perturb_at_t = 50

# --- r-value window (bins) ---
t_move_start = 50
t_move_end = 65

# --- snapshot plots ---
plot_ts = [48, 50, 55, 60, 70]
n_plot = len(plot_ts)

# --- simulation ---
n_duplications = 10
noise_scale = 1.0

# --- style ---
generate_plots = True
cmap = mpl.colors.ListedColormap(sns.color_palette("husl", n_colors=n_stim))

# --- run ---
run = False

In [ ]:
task_params_file = str(out_dir) + "/" + model_dirs[0] + "_task_params.pkl"
with open(task_params_file, "rb") as f:
    task_params = CPU_Unpickler(f).load()
bin_size = task_params["bin_size"]

u, _, labels, delay_ends = make_all_trials(
    task_params,
    dur=3.55,
    n_stim=n_stim,
    n_pos=n_pos,
    cue_dur=-1,
    bin_size=0.05,
    interval_dur="mean",
    delay_dur="mean",
)

u = np.concatenate([u] * n_duplications, axis=0)
labels = np.concatenate([labels] * n_duplications, axis=0)

In [ ]:
if run:
    rows = []

    for model_dir in model_dirs[:10]:
        model_dir = out_dir / Path(model_dir)
        name = model_dir.name

        vae, training_params, task_params = load_model(
            str(model_dir), load_encoder=True, backward_compat=False
        )

        row = df_basii.loc[df_basii["name"] == name]
        if row.empty:
            print(f"skip {name}: not in df_basii")
            continue

        A_comb_np = row["A_comb_np"].values[0]
        b_comb_np = row["b_comb_np"].values[0]
        rnn_orth = transformed_rnn(vae, A_comb_np, b_comb_np)
        print("model name:", name)
        print("macaque:", task_params["sessions"][0][5:10])

        if generate_plots:
            zs = rnn_orth.simulate(u)
            z_by_pos = [
                position_slice_at_time(zs, t_decode, i, n_pcs_time)
                for i in range(n_pos)
            ]
            c_by_pos = [labels[:, i] for i in range(n_pos)]
            plot_basis_2d_subspaces(
                z_by_pos, c_by_pos, cmap=cmap, n_stim=n_stim, n_pos=n_pos
            )

        # Start perturbation analysis
        # ------------
        Z = generate_w_perturb_x(rnn_orth, u=u, noise_scale=noise_scale)
        stats_all_pos = []
        for pos_ind, pos in enumerate(pos_perturb):
            z1 = z1s[pos_ind]
            z2 = z2s[pos_ind]
            labels_pos = labels[:, pos]
            unique_classes = np.unique(labels_pos)
            pert_dir = rnn_orth.W2[:, [z1, z2]]

            # box around class means in the 2D readout subspace (x @ pert_dir)
            class_means = np.array(
                [
                    (Z[labels_pos == c][:, [z1, z2], perturb_at_t]).mean(axis=0)
                    for c in unique_classes
                ]
            )
            # pert_z1 = np.array([np.min(class_means[:, 0]), np.max(class_means[:, 0])]) * 1.2
            # pert_z2 = np.array([np.min(class_means[:, 1]), np.max(class_means[:, 1])]) * 1.2

            pert_z1 = (
                np.array(
                    [np.min(Z[:, z1, perturb_at_t]), np.max(Z[:, z1, perturb_at_t])]
                )
                * 1.2
            )
            pert_z2 = (
                np.array(
                    [np.min(Z[:, z2, perturb_at_t]), np.max(Z[:, z2, perturb_at_t])]
                )
                * 1.2
            )

            goals = np.random.rand(u.shape[0], 2)
            goals[:, 0] = goals[:, 0] * (pert_z1[1] - pert_z1[0]) + pert_z1[0]
            goals[:, 1] = goals[:, 1] * (pert_z2[1] - pert_z2[0]) + pert_z2[0]

            Z_perturbed = generate_w_perturb_x(
                rnn_orth,
                u=u,
                noise_scale=noise_scale,
                perturb_at_t=perturb_at_t,
                perturb_weights=pert_dir,
                perturb_inds=list(position_latent_indices(n_pcs_time, pos)),
                goal=goals,
                goal_amp=1.0,
            )

            stats = compute_perturbation_distance_stats(
                Z,
                Z_perturbed,
                labels_pos,
                z1=z1,
                z2=z2,
                t_move_start=t_move_start,
                t_move_end=t_move_end,
            )

            if generate_plots:
                for highlight_cond in (0, None):
                    plot_perturbation_latent_snapshots_attractor(
                        Z,
                        Z_perturbed,
                        labels_pos,
                        z1=z1,
                        z2=z2,
                        cmap=cmap,
                        plot_ts=plot_ts,
                        highlight_cond=highlight_cond,
                        bin_size=bin_size,
                    )
                plot_distance_moved_vs_class_mean(
                    stats["distances_to_manifolds"],
                    stats["distance_moved"],
                    slope=stats["slope"],
                    intercept=stats["intercept"],
                    pearson_r=stats["pearson_r"],
                )
            print(stats)
            stats_all_pos.append(stats)
        # average stats over all positions
        stats = {}
        for key in stats_all_pos[0].keys():
            stats[key] = np.mean([s[key] for s in stats_all_pos], axis=0)
        print(stats)
        rows.append(
            {
                "name": name,
                "path": str(model_dir),
                "macaque": task_params["sessions"][0][5:10],
                "pearson_r": stats["pearson_r"],
                "p_val": stats["p_val"],
                "slope": stats["slope"],
                "n_valid_trials": stats["n_valid_trials"],
            }
        )

In [ ]:
if run:
    df = pd.DataFrame(rows)
    pickle.dump(df, open("../data/processed/df_perturb_attractor.pkl", "wb"))
else:
    df = pickle.load(open("../data/processed/df_perturb_attractor.pkl", "rb"))

In [ ]:
fig, ax = plot_pearson_r_by_macaque(
    df,
    box_w=0.3,
    box_h=0.6,
    ylims=(0.4, 0.7),
    save_path="../paper_figures/pearson_r_by_macaque.pdf",
);